# Laya V3.7 — Minimal-Memory Attention Necessity and Native-SDPA Speed Ablations

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V37_Attention_Necessity_Fast_SDPA_Ablation_Colab.ipynb)

## Why V3.6 failed

The uploaded **V3.6** report (Tesla T4, 200 decisions, source TRAIN case-disjoint audit) shows:

| Replacements | Agreement | Gold accuracy | Median full-model latency |
|---|---:|---:|---:|
| Original | 100% | 82.5% | **84.14 ms** |
| Route all 28 attention layers | 72% | 71.5% | 2721.75 ms |
| Slide all 28 attention layers | 72% | 67.0% | 90.65 ms |

**Critical confirmed bug in the V3.6 experimental design:** For \`transformers==4.57.6\`, \`ModernBertAttention\` uses \`attention_mask\` on full-attention layers, but on sliding layers it uses the separate **\`sliding_window_mask\`**. The V3.6 replacement module **ignored** the supplied sliding mask, and instead built a symmetric \`256\`-token window from the wrong mask. Hence the V3.6 "replace all sliding layers" comparison is not a trustworthy test of whether the original sliding attention is necessary. This V3.7 first checks exact *native-mask reconstruction* against the original sliding layer, refusing to run the screen if it fails.

### Objective: find the cheapest attention architecture that actually preserves pretrained decisions

**All candidate configurations reuse the original ModernBERT Wqkv, RoPE, Wo, native SDPA, norms and MLP weights.** We do **not** duplicate the 843-MB teacher, clone QKV tensors into 28 replacement modules, launch tiny per-query SDPA operations, or retain other model copies in GPU memory. A single frozen model is loaded once; only explicitly chosen attention modules are temporarily wrapped with a lightweight **mask-only** shim or a **zero-attention** diagnostic. All wrappers are uninstalled after every test.

Ablations **change one dimension at a time** using native mask semantics:
1. **Mask correctness:** native no-op wrapper, and rebuilt exact original local attention mask (same radius), must agree 100% with the original teacher.
2. **Where global attention is needed:** replace only ten full-attention layers with centered native-SDPA windows of radius 32, 64, 128, 256, optionally evaluate a full radius sweep.
3. **Which sliding windows are needed:** preserve ten global layers and shrink/widen only the original eighteen sliding layers.
4. **Combined architecture:** all 28 layers get a consistent local window, but native RoPE/kernel selection is preserved. Compare against (2) and (3) to isolate interactions.
5. **Attention necessity:** set the attention **residual contribution** to zero in full layers, sliding layers, or all layers. This measures *ablation importance*, not a trained attention-free network. An optional individual-layer ablation isolates the highest-impact layers; disabled by default.
6. **Efficiency breakdown:** measure warmed end-to-end T4 latency over matched real token lengths and batches, relative speed, and CUDA peak memory. Optional \`torch.compile\` candidate benchmarking is disabled by default because compilation may take minutes and can fall back.

### Leak-free selection

Probe and audit are disjoint at **case** level, drawn only from the source **TRAIN** split. Screen ~60 PROBE decisions per configuration. Rank by decision fidelity first, then latency. Evaluate **only a few preselected finalists** on up to 200 untouched AUDIT decisions; don't pick a final model from audit results. The official TEST split remains unused. A 200-decision audit has wide uncertainty; the report does not claim generalization or statistical equivalence.

### Acceptance criteria

- Baseline mask parity required: exact output replay in original full and sliding modules.
- A candidate must have teacher agreement ≥95%, gold-accuracy loss ≤1 percentage point, no unexpected numerical failure, and warmed end-to-end latency **less than native on matched inputs**.
- If none passes, **native attention wins**, and V3.7 gives a ranked engineering profile rather than declaring an unsupported speedup.

**Implementation footprint:** the notebook deliberately strips obsolete Colab widget output metadata, uses a single pretrained model, and includes peak allocated CUDA memory in its per-variant measurements. The exact-sliding-mask reconstruction is a **correctness control**, not a candidate architecture for deployment.


In [ ]:
#@title 1. Install pinned dependencies and check GPU
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"
import torch
assert torch.cuda.is_available(),"Select Runtime > Change runtime type > GPU"
print("PyTorch",torch.__version__,"CUDA",torch.version.cuda,
      "GPU",torch.cuda.get_device_name(0))


In [ ]:
#@title 2. Imports, frozen Laya teacher and experiment configuration
from pathlib import Path
import copy, json, math, random, time
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

from tinycenn_lm.decision_v25 import layer_kinds, MultiscaleCeNN
from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence,
    collate_items, normalize_question,
)
from tinycenn_lm.decision_training import require_finite

SEED = 2037
SOURCE_MODEL = "convaiinnovations/laya-typed-decisions" #@param {type:"string"}
LAYER = 15 #@param {type:"integer"}
TARGET_LAYERS = (LAYER,)
TRAIN_CASES = 80 #@param {type:"integer"}
DEV_CASES = 160 #@param {type:"integer"}
MAX_LEN = 512 #@param {type:"integer"}
BATCH = 3 #@param {type:"integer"}
STEPS = 400 #@param {type:"integer"}
REFINE_STEPS = 1200 #@param {type:"integer"}
REFINE_TOP_K = 2 #@param {type:"integer"}
TRAIN_CACHE_BATCHES = 1 #@param {type:"integer"}
PROBE_CACHE_BATCHES = 2 #@param {type:"integer"}
AUDIT_CACHE_BATCHES = 2 #@param {type:"integer"}
CHECK_EVERY = 100 #@param {type:"integer"}
OUTPUT_DIR = Path("/content/Laya_V37_FastAttentionAblation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert DEV_CASES >= 4 and TRAIN_CACHE_BATCHES > 0
assert REFINE_TOP_K >= 1 and TRAIN_CACHE_BATCHES > 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Use a CUDA GPU runtime in Google Colab.")
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda")
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16

source_dir = Path(snapshot_download(
    SOURCE_MODEL, allow_patterns=[
        "rl_agent_config.json", "model.safetensors", "encoder/*", "tokenizer/*",
    ],
))
teacher, source_cfg = build_checkpoint_model(source_dir)
# student_eval not built: use a single model and temporarily swap wrappers
tokenizer = AutoTokenizer.from_pretrained(str(source_dir / "tokenizer"))
teacher.to(device).eval().requires_grad_(False)
# single teacher model only
teacher.encoder.config.reference_compile = False
# one model; reference_compile disabled above

kinds = layer_kinds(teacher.encoder)
full_layers = [i for i, kind in enumerate(kinds) if kind == "full_attention"]
assert LAYER in full_layers, (LAYER, full_layers)
print("GPU:", torch.cuda.get_device_name(0), "AMP:", amp_dtype)
print("Source:", SOURCE_MODEL)
print("Target:", LAYER, "all full attention layers:", full_layers)
#@title V3.7 controls
PROBE_DECISIONS = 60 #@param {type:"integer"}
AUDIT_DECISIONS = 200 #@param {type:"integer"}
BENCHMARK_REPEATS = 6 #@param {type:"integer"}
MAX_FINALISTS = 3 #@param {type:"integer"}
RUN_SINGLE_LAYER_SCAN = False #@param {type:"boolean"}
RUN_EXTENDED_WINDOWS = False #@param {type:"boolean"}
RUN_COMPILE_PILOT = False #@param {type:"boolean"}
assert PROBE_DECISIONS>=20 and AUDIT_DECISIONS>=20
assert MAX_FINALISTS>=1 and BENCHMARK_REPEATS>=3
ALL_LAYERS=list(range(len(kinds)))
FULL_LAYERS=[i for i,k in enumerate(kinds) if k=="full_attention"]
SLIDING_LAYERS=[i for i in ALL_LAYERS if i not in FULL_LAYERS]
assert len(ALL_LAYERS)==28 and len(FULL_LAYERS)==10 and len(SLIDING_LAYERS)==18
ORIGINAL_LOCAL_RADIUS=int(teacher.encoder.config.local_attention)//2
print("Original native local radius:",ORIGINAL_LOCAL_RADIUS)
print("Full:",FULL_LAYERS,"Sliding:",SLIDING_LAYERS)
print("One loaded model only. Frozen weights, native SDPA. TEST untouched.")


In [ ]:
#@title 3. Leak-free train/dev data and teacher I/O helpers
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
rng = random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES + DEV_CASES <= len(rows)
train_rows = rows[:TRAIN_CASES]
dev_rows = rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
# Split at the CASE level before creating multiple question items per case.
probe_rows = dev_rows[:len(dev_rows) // 2]
audit_rows = dev_rows[len(dev_rows) // 2:]
probe_items = make_items_from_rows(probe_rows, SEED + 1, True)
audit_items = make_items_from_rows(audit_rows, SEED + 2, True)
dev_items = probe_items # backward-compatible alias for training helper use
assert train_items and probe_items and audit_items
assert {str(r["id"]) for r in probe_rows}.isdisjoint({str(r["id"]) for r in audit_rows})

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

def _snapshot_attention_argument(value):
    """Retain the exact forward inputs for ModernBERT SDPA replay/timing."""
    if torch.is_tensor(value):
        return value.detach()
    if isinstance(value, tuple):
        return tuple(_snapshot_attention_argument(v) for v in value)
    if isinstance(value, list):
        return [_snapshot_attention_argument(v) for v in value]
    if isinstance(value, dict):
        return {k: _snapshot_attention_argument(v) for k, v in value.items()}
    return value

def _attention_output_tensor(result):
    """ModernBERT variants return either tensor or (tensor, optional weights)."""
    return result[0] if isinstance(result, (tuple, list)) else result

def attn_io(model, layer_idx, b):
    layer = model.encoder.layers[layer_idx]
    got = {}
    def pre(mod, args, kwargs):
        # The encoder passes more than x/position_embeddings/attention_mask:
        # position_ids, sliding_window_mask and optional flags are required by
        # transformers 4.57.6 SDPA. Never infer these for benchmarking.
        got["call_args"] = _snapshot_attention_argument(tuple(args))
        got["call_kwargs"] = _snapshot_attention_argument(dict(kwargs))
        got["x"] = args[0].detach()
        got["pos"] = _snapshot_attention_argument(kwargs.get("position_embeddings"))
        got["mask"] = _snapshot_attention_argument(kwargs.get("attention_mask"))
    def post(mod, args, kwargs, out):
        got["y"] = _attention_output_tensor(out).detach()
    h1 = layer.attn.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = layer.attn.register_forward_hook(post, with_kwargs=True)
    try:
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
    finally:
        h1.remove()
        h2.remove()
    return got

def cache_teacher_io(layer_idx, items, batches, offset=0):
    cache = []
    for i in range(batches):
        b = batch(items, offset + i * BATCH, BATCH)
        got = attn_io(teacher, layer_idx, b)
        cache.append({
            "x": got["x"],
            "pos": got["pos"],
            "mask": got["mask"],
            "call_args": got["call_args"],
            "call_kwargs": got["call_kwargs"],
            "y": got["y"],
            "valid": b["attention_mask"].bool(),
        })
    return cache

def local_metrics_from_cache(repl, cache):
    repl.eval()
    diff_sum = power_sum = cos_sum = 0.0
    tokens = 0
    with torch.no_grad():
        for entry in cache:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                pred, _ = repl(
                    entry["x"],
                    position_ids=entry["call_kwargs"].get("position_ids"),
                    attention_mask=entry["mask"],
                )
            require_finite(pred, "V3.7 local probe")
            valid = entry["valid"]
            m = valid[:, :, None].float()
            d = pred.shape[-1]
            diff_sum += float((((pred.float() - entry["y"].float()) ** 2) * m).sum())
            power_sum += float(((entry["y"].float() ** 2) * m).sum())
            count = int(valid.sum())
            cos_sum += float(
                F.cosine_similarity(
                    pred.float()[valid],
                    entry["y"].float()[valid],
                    dim=-1,
                ).sum()
            )
            tokens += count
    nmse = diff_sum / max(power_sum, 1e-8)
    cosine = cos_sum / max(tokens, 1)
    score = nmse + 1.10 * (1.0 - cosine)
    return {"nmse": nmse, "cosine": cosine, "score": score, "tokens": tokens}

print("train/probe/audit decisions:", len(train_items), len(probe_items), len(audit_items))
print("unique train/probe/audit cases:", len(train_rows), len(probe_rows), len(audit_rows))
assert len(probe_items) >= PROBE_CACHE_BATCHES * BATCH, "Increase DEV_CASES or reduce PROBE_CACHE_BATCHES"
assert len(audit_items) >= AUDIT_CACHE_BATCHES * BATCH, "Increase DEV_CASES or reduce AUDIT_CACHE_BATCHES"
print("official test remains untouched")


In [ ]:
#@title 4. Single-model mask-preserving hooks: original full vs sliding SDPA semantics
from functools import lru_cache
from contextlib import contextmanager

# Original ModernBERT 4.57.6 SDPA consumes separate masks:
# full attention -> attention_mask; sliding attention -> sliding_window_mask.
# Preserve the native module itself: no QKV/Wo/RoPE cloning or alternative SDPA.
# The original encoder creates both 4D additive masks before visiting layers.

@lru_cache(maxsize=72)
def cached_band(length,radius,dev):
    ar=torch.arange(length,device=torch.device(dev))
    return ((ar[:,None]-ar[None,:]).abs()<=radius)[None,None,:,:]

def narrowed_mask(original_global,radius,T):
    if original_global is None:
        raise RuntimeError("Expected original encoder's 4D additive global mask.")
    if original_global.ndim!=4 or original_global.shape[-2:]!=(T,T):
        raise RuntimeError(f"Unexpected native global mask {original_global.shape}")
    band=cached_band(T,int(radius),str(original_global.device))
    if original_global.dtype==torch.bool:
        return original_global & band
    return original_global.masked_fill(~band,torch.finfo(original_global.dtype).min)

class AttentionMaskAblation(nn.Module):
    """Zero-copy (parameters owned by original module) native-SDPA mask shim."""
    def __init__(self,original,original_kind,mode="band",radius=128):
        super().__init__()
        self.original=original
        self.kind=original_kind
        self.mode=mode
        self.radius=int(radius)

    def forward(self,hidden_states,*args,**kwargs):
        if self.mode=="zero":
            # Skip only attention residual: encoder FFN, norms, classifier stay.
            return (torch.zeros_like(hidden_states),)
        if self.mode=="passthrough":
            return self.original(hidden_states,*args,**kwargs)
        original_global=kwargs.get("attention_mask")
        original_slide=kwargs.get("sliding_window_mask")
        if self.mode=="band":
            changed=narrowed_mask(original_global,self.radius,hidden_states.shape[1])
            if self.kind=="full_attention":
                kwargs=dict(kwargs,attention_mask=changed)
            else:
                kwargs=dict(kwargs,sliding_window_mask=changed)
        elif self.mode=="global":
            if self.kind=="full_attention":
                return self.original(hidden_states,*args,**kwargs)
            kwargs=dict(kwargs,sliding_window_mask=original_global)
        else:
            raise ValueError(self.mode)
        return self.original(hidden_states,*args,**kwargs)

original_attention=[layer.attn for layer in teacher.encoder.layers]

@contextmanager
def temporary_attention(spec):
    """spec maps layer index -> (mode, radius); safely restore after each run."""
    assert all(layer.attn is original_attention[i]
               for i,layer in enumerate(teacher.encoder.layers)),(
        "Aborted: some native attention module already replaced"
    )
    try:
        for idx,(mode,radius) in spec.items():
            teacher.encoder.layers[idx].attn=AttentionMaskAblation(
                original_attention[idx],kinds[idx],mode=mode,radius=radius
            )
        teacher.eval()
        yield teacher
    finally:
        for idx,module in enumerate(original_attention):
            teacher.encoder.layers[idx].attn=module
        teacher.eval()
        assert all(layer.attn is original_attention[i]
                   for i,layer in enumerate(teacher.encoder.layers))

def configure_full(radius):
    return {i:("band",radius) for i in FULL_LAYERS}
def configure_sliding(radius):
    return {i:("band",radius) for i in SLIDING_LAYERS}
def configure_both(radius):
    return {i:("band",radius) for i in ALL_LAYERS}
def configure_skip(indices):
    return {i:("zero",0) for i in indices}

def spec_description(spec):
    return {str(i):dict(mode=v[0],radius=v[1],original_kind=kinds[i])
            for i,v in sorted(spec.items())}
print("No-clone mask-only shim ready. Original native layers retained by default.")


In [ ]:
#@title 5. Mandatory mask-correctness preflight on original full and sliding layers
# Run before any experiment: a bad native mask would invalidate every result.
# Source ModernBERT uses distance <= config.local_attention // 2,
# NOT a centered 256-token total-width convention.

b=batch(probe_items,0,BATCH)
for i in (FULL_LAYERS[0],SLIDING_LAYERS[0],SLIDING_LAYERS[-1]):
    rec=attn_io(teacher,i,b)
    original_global=rec["call_kwargs"]["attention_mask"]
    original_sliding=rec["call_kwargs"]["sliding_window_mask"]
    assert original_global.ndim==4 and original_sliding.ndim==4
    T=rec["x"].shape[1]
    rebuilt=narrowed_mask(original_global,ORIGINAL_LOCAL_RADIUS,T)
    # The EXACT original mask must be reconstructible from the global mask.
    assert torch.equal(rebuilt,original_sliding),(
        "Original sliding mask mismatch — do not proceed",i,
        float((rebuilt.float()-original_sliding.float()).abs().max())
    )
    for mode,radius in (("passthrough",ORIGINAL_LOCAL_RADIUS),
                        ("band",ORIGINAL_LOCAL_RADIUS)):
        # A full-attention layer is not expected to be identical after
        # changing its mask. Native parity on full: passthrough only.
        if kinds[i]=="full_attention" and mode=="band":continue
        wrapper=AttentionMaskAblation(
            original_attention[i],kinds[i],mode=mode,radius=radius
        )
        with torch.inference_mode(),torch.autocast(device_type="cuda",dtype=amp_dtype):
            replay=_attention_output_tensor(wrapper(
                *rec["call_args"],**rec["call_kwargs"]
            ))
        diff=float((replay.float()-rec["y"].float()).abs().max())
        assert diff<1e-2,(i,mode,diff)
        print("Native SDPA masking parity PASS",kinds[i],
              "layer",i,mode,"maxΔ",diff)
print("Mask path + exact original sliding radius verified.")


In [ ]:
#@title 6. Cache BASELINE distributions once (no duplicate 843-MB model)
@torch.inference_mode()
def predictions(items,limit):
    out=[]
    for first in range(0,min(limit,len(items)),BATCH):
        segment=items[first:min(first+BATCH,limit)]
        batch_inputs=direct_batch(segment)
        with torch.autocast(device_type="cuda",dtype=amp_dtype):
            scores,_=teacher(**batch_inputs)
        allowed=batch_inputs["marker_mask"].bool()
        p=torch.softmax(scores.float().masked_fill(~allowed,-1e9),dim=-1)
        for j,item in enumerate(segment):
            out.append(dict(
                probabilities=p[j,:].cpu(),
                decision=int(p[j].argmax()),
                gold=int(item["gold_index"]),
                case_id=item["case_id"],qid=item["qid"],
                token_length=len(item["ids"])
            ))
    return out

probe_ref=predictions(probe_items,PROBE_DECISIONS)
audit_ref=predictions(audit_items,AUDIT_DECISIONS)
assert len(probe_ref)==PROBE_DECISIONS and len(audit_ref)==AUDIT_DECISIONS
assert {x["case_id"] for x in probe_ref}.isdisjoint(
    {x["case_id"] for x in audit_ref}
)
print("Cached",len(probe_ref),"probe and",len(audit_ref),
      "audit native predictions on CPU; no other model allocated.")
print("Original teacher gold accuracy, audit:",
      sum(r["decision"]==r["gold"] for r in audit_ref)/len(audit_ref))


In [ ]:
#@title 7. Fast decision + CUDA-event benchmarks (true pretrained whole-model)
@torch.inference_mode()
def measure_quality(items,reference):
    n=min(len(items),len(reference))
    agree=correct=teach_correct=0
    js_total=0.0
    differences=[]
    for off in range(0,n,BATCH):
        segment=items[off:min(n,off+BATCH)]
        b=direct_batch(segment)
        with torch.autocast(device_type="cuda",dtype=amp_dtype):
            scores,_=teacher(**b)
        mask=b["marker_mask"].bool()
        predicted=torch.softmax(scores.float().masked_fill(~mask,-1e9),dim=-1).cpu()
        for j,item in enumerate(segment):
            baseline=reference[off+j]
            s=predicted[j]
            p=baseline["probabilities"]
            assert len(s)==len(p),(len(s),len(p))
            # Only allowed marker entries are used; padded option probs are 0.
            avg=.5*(s+p)
            term=.5*((s*(s.clamp_min(1e-8).log()-avg.clamp_min(1e-8).log())).sum()+
                     (p*(p.clamp_min(1e-8).log()-avg.clamp_min(1e-8).log())).sum())
            js_total+=float(term)
            guess=int(s.argmax())
            correct+=int(guess==item["gold_index"])
            teach_correct+=int(baseline["decision"]==item["gold_index"])
            agree+=int(guess==baseline["decision"])
            if guess!=baseline["decision"]:
                differences.append(dict(case=item["case_id"],qid=item["qid"],
                                        length=len(item["ids"]),
                                        teacher_correct=baseline["decision"]==item["gold_index"],
                                        student_correct=guess==item["gold_index"]))
    return dict(
        decisions=n,teacher_agreement=agree/max(n,1),
        teacher_gold_accuracy=teach_correct/max(n,1),
        student_gold_accuracy=correct/max(n,1),
        gold_accuracy_drop=(teach_correct-correct)/max(n,1),
        mean_js=js_total/max(n,1),
        differing_decisions=differences[:20]
    )

@torch.inference_mode()
def cuda_latency(fun,warmup=3,repeats=BENCHMARK_REPEATS):
    with torch.autocast(device_type="cuda",dtype=amp_dtype):
        for _ in range(warmup):fun()
        torch.cuda.synchronize()
        values=[]
        for _ in range(repeats):
            a=torch.cuda.Event(enable_timing=True)
            z=torch.cuda.Event(enable_timing=True)
            a.record();fun();z.record();z.synchronize()
            values.append(float(a.elapsed_time(z)))
    return float(np.median(values))

# Keep the same batch objects and length distribution for every candidate.
timing_batches=[
    batch(probe_items,0,BATCH),
    batch(probe_items,5,BATCH),
    batch(probe_items,10,BATCH),
]
timing_lengths=[int(x["input_ids"].shape[1]) for x in timing_batches]
torch.cuda.reset_peak_memory_stats()
native_times=[
    cuda_latency(lambda b=b:teacher(**b))
    for b in timing_batches
]
native_peak_mib=float(torch.cuda.max_memory_allocated()/1024**2)
native_median=float(np.median(native_times))
print("Matched real-sequence timing lengths:",timing_lengths)
print("Native reference times:",native_times,"median",native_median,"ms")

def candidate_latency():
    torch.cuda.reset_peak_memory_stats()
    timings=[cuda_latency(lambda b=b:teacher(**b))
             for b in timing_batches]
    peak_mib=float(torch.cuda.max_memory_allocated()/1024**2)
    return float(np.median(timings)),peak_mib

def run_candidate(spec,which="probe",include_latency=True):
    items,ref=(probe_items,probe_ref) if which=="probe" else (audit_items,audit_ref)
    with temporary_attention(spec):
        quality=measure_quality(items,ref)
        ms,peak=(candidate_latency() if include_latency else (None,None))
    return dict(quality=quality,model_ms=ms,
                peak_allocated_gpu_mib=peak,
                native_peak_allocated_gpu_mib=native_peak_mib,
                native_ms=native_median,
                speedup=native_median/max(ms,1e-9) if ms is not None else None,
                affected_layers=len(spec))

# Strict no-op baseline must not change a single decision.
with temporary_attention({i:("passthrough",0)
                          for i in [FULL_LAYERS[0],SLIDING_LAYERS[0]]}):
    none=measure_quality(probe_items,probe_ref)
assert none["teacher_agreement"]==1.0 and none["mean_js"]<1e-6,none
print("Single-model no-op correctness PASS",none["teacher_agreement"])


In [ ]:
#@title 8. PROBE ablation: full/sliding masks, local radius, attention removal
# Screen ONLY probe data. Never use the audit split to pick configurations.
experiments={
    "native":{},
    "sliding_rebuild_exact":configure_sliding(ORIGINAL_LOCAL_RADIUS),
    "full_band_r32":configure_full(32),
    "full_band_r64":configure_full(64),
    "full_band_r128":configure_full(128),
    "full_band_r256":configure_full(256),
    "sliding_band_r32":configure_sliding(32),
    "sliding_band_r128":configure_sliding(128),
    "sliding_band_r256":configure_sliding(256),
    "all_band_r64":configure_both(64),
    "all_band_r128":configure_both(128),
    "all_band_r256":configure_both(256),
    "zero_full_attention":configure_skip(FULL_LAYERS),
    "zero_sliding_attention":configure_skip(SLIDING_LAYERS),
    "zero_all_attention":configure_skip(ALL_LAYERS),
}
if RUN_EXTENDED_WINDOWS:
    experiments.update({
        "full_band_r16":configure_full(16),
        "full_band_r384":configure_full(384),
        "sliding_band_r16":configure_sliding(16),
        "sliding_global":{i:("global",0) for i in SLIDING_LAYERS},
        "zero_even_sliding":configure_skip(SLIDING_LAYERS[::2]),
        "zero_odd_sliding":configure_skip(SLIDING_LAYERS[1::2]),
    })
probe_results={}
for name,spec in experiments.items():
    print("Running probe:",name,"replaced layers",len(spec))
    result=run_candidate(spec,which="probe")
    probe_results[name]=result
    q=result["quality"]
    print(f"  agree={q['teacher_agreement']:.4f}, "
          f"gold_drop={q['gold_accuracy_drop']:+.4f}, "
          f"JS={q['mean_js']:.5f}, "
          f"latency={result['model_ms']:.2f}ms, "
          f"speedup={result['speedup']:.3f}x")
    torch.cuda.empty_cache()
assert abs(probe_results["sliding_rebuild_exact"]["quality"]["teacher_agreement"]-1)<1e-8,(
    "Sliding mask reconstruction changed original decisions unexpectedly!"
)
assert probe_results["sliding_rebuild_exact"]["quality"]["mean_js"]<1e-5,(
    "Sliding mask reconstruction changed original logits; investigate immediately."
)


In [ ]:
#@title 9. Optional individual-layer necessity screen — set toggle first
layer_drop={}
if RUN_SINGLE_LAYER_SCAN:
    print("Single-layer ablation uses PROBE only; may take several minutes.")
    for i in ALL_LAYERS:
        row=run_candidate(configure_skip([i]),which="probe",include_latency=False)
        layer_drop[str(i)]=dict(kind=kinds[i],quality=row["quality"])
        print(f"L{i:02d} {kinds[i]:<18} "
              f"agreement={row['quality']['teacher_agreement']:.4f}, "
              f"gold_drop={row['quality']['gold_accuracy_drop']:+.4f}")
else:
    print("Individual-layer deletion scan disabled for fast initial run.")


In [ ]:
#@title 10. Preselect finalists using ONLY PROBE fidelity and latency
# For 60 decisions, one disagreement is 1.67 percentage points, so use a
# RELAXED 3pp probe gold gate; final AUDIT uses the 1pp strict gate.
# Select by agreement, gold loss, JS, and only then speed.
ranked=[]
for name,result in probe_results.items():
    if name in ("native","sliding_rebuild_exact"):
        continue
    q=result["quality"]
    if q["teacher_agreement"]>=.95 and q["gold_accuracy_drop"]<=.03:
        ranked.append((name,result))
ranked.sort(key=lambda pair:(
    -pair[1]["quality"]["teacher_agreement"],
    pair[1]["quality"]["gold_accuracy_drop"],
    pair[1]["quality"]["mean_js"],
    pair[1]["model_ms"]
))
finalists=["native"]+[n for n,_ in ranked[:MAX_FINALISTS]]
# Also audit the fastest non-native candidate (even if inaccurate), purely
# as a diagnostic; it cannot become a recommended model without passing gates.
fastest=min((n for n in experiments if n!="native"),
            key=lambda n:probe_results[n]["model_ms"])
if fastest not in finalists:
    finalists.append(fastest)
print("Probe-only finalists:",finalists)
print("Fastest change regardless of quality (diagnostic only):",fastest)
print("Eligible probe candidates:",[n for n,_ in ranked])


In [ ]:
#@title 11. Held-out AUDIT of preselected finalists, with matched real GPU latency
audit_results={}
for name in finalists:
    result=run_candidate(experiments[name],which="audit")
    audit_results[name]=result
    q=result["quality"]
    gates={
        "agreement_ge_95":q["teacher_agreement"]>=.95,
        "accuracy_loss_le_1pp":q["gold_accuracy_drop"]<=.01,
        "faster_than_native":result["speedup"]>1.0
    }
    gates["all_pass"]=all(gates.values())
    result["gates"]=gates
    print(f"{name:>24}: agree={q['teacher_agreement']:.4f}, "
          f"gold={q['student_gold_accuracy']:.4f}, "
          f"drop={q['gold_accuracy_drop']:+.4f}, "
          f"time={result['model_ms']:.2f}ms, speedup={result['speedup']:.3f}x "
          f"PASS={gates['all_pass']}")
valid=[n for n,row in audit_results.items()
       if n!="native" and row["gates"]["all_pass"]]
if valid:
    print("Acceptable alternatives observed:",valid)
else:
    print("No alternative met all gates: native ModernBERT remains safest and fastest acceptable baseline.")


In [ ]:
#@title 12. Optional torch.compile pilot for one preselected candidate
compile_results=None
if RUN_COMPILE_PILOT:
    # Compilation on Tesla T4 and dynamic token shapes can take minutes.
    # Train/test decisions are never reselected by this optional pilot.
    candidate_name=next((n for n in finalists if n!="native"),"native")
    spec=experiments[candidate_name]
    with temporary_attention(spec):
        torch._dynamo.reset()
        try:
            compiled=torch.compile(teacher,mode="reduce-overhead",
                                   fullgraph=False,dynamic=False)
            single=timing_batches[0]
            for _ in range(3):
                with torch.inference_mode(),torch.autocast("cuda",dtype=amp_dtype):
                    compiled(**single)
            ms=cuda_latency(lambda:compiled(**single),warmup=2,repeats=5)
            with torch.inference_mode(),torch.autocast("cuda",dtype=amp_dtype):
                raw=teacher(**single)[0]
                fast=compiled(**single)[0]
            assert torch.allclose(raw.float(),fast.float(),atol=.02,rtol=.005)
            compile_results=dict(model=candidate_name,ms=ms,
                                 baseline_batch_ms=native_times[0],
                                 speedup_vs_native=native_times[0]/ms)
            print("Compiled candidate:",compile_results)
        except Exception as ex:
            compile_results=dict(model=candidate_name,error=repr(ex))
            print("torch.compile pilot unsupported:",repr(ex))
        finally:
            if "compiled" in locals():del compiled
            torch.cuda.empty_cache()
else:
    print("No compilation attempted; opt in after screening. Native SDPA is already fused.")


In [ ]:
#@title 13. Save reproducible V3.7 ablation matrix and plots
from datetime import datetime,timezone
import matplotlib.pyplot as plt
payload=dict(
    version="3.7-minimal-memory-native-sdpa-attention-ablation",
    timestamp=datetime.now(timezone.utc).isoformat(),
    source_model=SOURCE_MODEL,
    gpu=torch.cuda.get_device_name(0),
    torch_version=torch.__version__,
    seed=SEED,official_test_used=False,
    all_layers=len(ALL_LAYERS),
    full_layers=FULL_LAYERS,sliding_layers=SLIDING_LAYERS,
    original_local_radius=ORIGINAL_LOCAL_RADIUS,
    probe_decisions=len(probe_ref),audit_decisions=len(audit_ref),
    native_latency_ms=native_median,
    native_peak_allocated_gpu_mib=native_peak_mib,
    timing_batch_lengths=timing_lengths,
    original_teacher_on_gpu_copies=1,
    pretrained_qkv_rope_wo_preserved=True,
    native_sdpa_preserved=True,
    probe_ablation={name:dict(spec=spec_description(experiments[name]),**result)
                   for name,result in probe_results.items()},
    optional_single_layer_ablation=layer_drop,
    finalists_selected_using_probe_only=finalists,
    audit_only_finalists=audit_results,
    compilation_pilot=compile_results,
    limitations=[
        "Zero attention is an ablation of residual contribution, not a trained replacement.",
        "Original ModernBERT sliding mask is explicitly passed to sliding layers.",
        "Banded 4D masks may still use dense SDPA kernels; speed depends on backend.",
        "GPU one-model benchmark measures warmed inference, not compile/loading time.",
        "200 decisions have statistical uncertainty; official test remains unused.",
        "None of the full network's MLP/head parameters or pretrained attention weights are pruned.",
        "A true smaller model would require structured channel, head or layer pruning with distillation and re-evaluation."
    ]
)
out=OUTPUT_DIR/"v37_native_attention_necessity_ablation_report.json"
out.write_text(json.dumps(payload,indent=2,default=str),encoding="utf-8")
print("Saved:",out)
names=list(probe_results)
timings=[probe_results[n]["model_ms"] for n in names]
quality=[probe_results[n]["quality"]["teacher_agreement"] for n in names]
fig,axes=plt.subplots(2,1,figsize=(10,8))
axes[0].bar(range(len(names)),timings)
axes[0].axhline(native_median,linestyle="--",label="Original native")
axes[0].set_ylabel("CUDA model latency (ms)")
axes[0].set_title("Probe: speed (lower is better)")
axes[0].legend()
axes[1].bar(range(len(names)),quality)
axes[1].axhline(.95,linestyle="--",label="Agreement gate")
axes[1].set_ylim(0,1.05)
axes[1].set_ylabel("Teacher decision agreement")
axes[1].set_xticks(range(len(names)),names,rotation=65,ha="right")
axes[1].legend()
plt.tight_layout()
plt.show()
print("FINAL:",[(n,round(v["speedup"],3),v["gates"]) for n,v in audit_results.items()])
